In [ ]:
# Реализуйте пакетный градиентный спуск (**Batch Gradient Descent**) с ранней остановкой (**Early Stopping**) 
# для Softmax Regression без использования Scikit-Learn, только с помощью NumPy. 
# Используйте его для задачи классификации, например, на датасете Iris.

In [2]:
# Detect iris virginica
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
print(list(iris), "\n")
print(iris.data.head(3),  "\n")
# instances are not shuffled
print(iris.target.head(3),  "\n")
print(iris.target_names,  "\n")
# чашелистик, лепесток

['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'] 

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2 

0    0
1    0
2    0
Name: target, dtype: int32 

['setosa' 'versicolor' 'virginica'] 



In [59]:
# Detect iris virginica
import numpy as np
import matplotlib.pyplot as plt

X = iris.data.values
y = iris.target
# split data 80:20
def split_data(x, y):
    # X_train, X_test, y_train, y_test
    train_end = int(len(X) / 100 * 80)
    valid_start = train_end+1
    X_train = x[0:train_end]
    y_train = x[0:train_end]
    X_test = x[valid_start:len(X)]
    y_test = y[valid_start:len(X)]
    return X_train,y_train, X_test, y_test

X_train, y_train, X_test, y_test = split_data(X,y)
print(len(X_train))
print(len(y_train))
print(len(X_test))
print(len(y_test))

virginica_value_test = X[y][:1][0]
not_virginica_value_test = X[~y][:1][0]
print("Example with petal width is virginica",virginica_value_test)
print("Example with petal width is not virginica",not_virginica_value_test)


120
120
29
29
Example with petal width is virginica [5.1 3.5 1.4 0.2]
Example with petal width is not virginica [5.9 3.  5.1 1.8]


In [ ]:
# Batch gradient
# Early stopping
# SoftmaxRegression


import numpy as np
import math

def generateXvector(X):
    """
    функция добавляет единицы к данным, чтобы можно было вычислять прогноз одним матричным умножением.
    """
    # np.c_ склеивает массивы по столбцам.
    # np.ones Добавляем столбец единиц слева от X
    vectorX = np.c_[np.ones((len(X), 1)), X]
    return vectorX


def theta_init(X):
    # Independent variables (X) — площадь и количество комнат. Это то, что мы подаём на вход модели.
    # Dependent variable (y) — цена. Это то, что мы хотим предсказать, и она зависит от входных признаков.
    # Количество элементов в theta = количество признаков + 1 (для свободного члена theta).
    # X = [ [50, 2], [2,3], [4,4]    # 2 признака]
    theta = np.random.randn(len(X[0]) + 1, 1)
    return theta


def sigmoid_function(X):
    return 1 / (1 + np.exp(-X))


class Logistics_Regression:

    def __init__(self, learningrate, iterations, withEarlyStopping, EARLY_STOP_MAX_I = 30):
        self.learningrate = learningrate
        self.iterations = iterations
        self.withEarlyStopping = withEarlyStopping
        self.minError = None 
        self.bestTheta = None
        self.EARLY_STOP_MAX_I = EARLY_STOP_MAX_I
        self.earlyStopIteration = 0
        self.cost_lst = []

    def _score(self, x, thetas):
        scores = []
        for _, x_val in enumerate(x):
            row = []
            for _,theta_k in enumerate(thetas):
                row.append(theta_k @ x_val)
            scores.append(row)
        return scores

    def softmax(self, x):
        S = self._score(x, [[0.1, 0.2, 0.3, 0.4],[0.3, 0.7, 0.5, 0.9],[0.5, 0.1, 0.4, 0.7]])
        P = []
        for i, score in enumerate(S):
            sum_exp = 0
            for j, value in enumerate(score):			
                sum_exp += math.exp(value)
                row = [] 
                for j, value in enumerate(score):
                    row.append(math.exp(value) /  sum_exp)
                P.append(row)
        return P

    def loss(P, Y_onehot):           # средняя cross entropy
        ...
        

    def fit(self, X, y):
        y_new = np.reshape(y, (len(y), 1))
        vectorX = generateXvector(X)
        theta = theta_init(X)
        m = len(X)
        for i in range(self.iterations):
            # TODO разобрать
            gradients = 2 / m * vectorX.T @ (sigmoid_function(vectorX @ (theta)) - y_new)
            theta = theta - self.learningrate  * gradients
            y_pred = sigmoid_function(vectorX @ theta)
            cost_value = -np.sum(y_new.T @ np.log(y_pred) + (1 - y_new).T @ np.log(1 - y_pred)) / len(y_pred)
            rounded_cost_value = round(cost_value,2)
            # Calculate the loss for each training instance
            self.cost_lst.append(cost_value)

            if self.minError is None or rounded_cost_value < self.minError:
                self.minError = rounded_cost_value
                self.bestTheta = theta
                self.earlyStopIteration = 0
            else:
                self.earlyStopIteration += 1
                if self.withEarlyStopping and self.earlyStopIteration > self.EARLY_STOP_MAX_I:
                    print("stopped on", i)
                    break



    def predict(self,val):
      vectorX = generateXvector(val)
      return sigmoid_function(vectorX @ self.bestTheta)

classifier = Logistics_Regression(0.2, 500, True)
# classifier.fit(X,y)
# print(classifier.minError, classifier.bestTheta,"\n")


# print("Is virginica?",classifier.predict(virginica_value_test) > 0.5)
# print("Is not?",classifier.predict(not_virginica_value_test) > 0.5)

print(classifier.softmax(X))

[[1.0, 23.336064580942697, 6.619368681043075], [0.04109127820046504, 0.958908721799535, 0.27199831998418633], [0.032304506660807454, 0.7538600536920964, 0.21383543964709628], [1.0, 17.461526936579972, 6.423736771429133], [0.05416670048123612, 0.9458332995187639, 0.3479526256683046], [0.04018442447439923, 0.7016814103906857, 0.25813416513491516], [1.0, 18.17414536944306, 5.754602676005729], [0.05215356307841774, 0.9478464369215823, 0.30012303365429627], [0.04011432897379573, 0.7290436461674253, 0.23084202485877905], [1.0, 17.637018199837325, 5.697343422671992], [0.05365665200717186, 0.9463433479928282, 0.30570037339566053], [0.04109415383533212, 0.7247783391006674, 0.23412750706400043], [1.0, 24.046753552064505, 6.296538261026658], [0.03992533395281378, 0.9600746660471863, 0.25139139281815864], [0.031904753526313716, 0.7672057451866269, 0.20088950128705949], [1.0, 35.51659315162849, 7.845969810318453], [0.027384810950125675, 0.9726151890498744, 0.21486039997596423], [0.02254152901079620